# Governance & Security — ACLs, Masking, Row Filters, ABAC

## Reference domain

The bank has three principal groups consuming Fintech data:

- **`analysts`** — see aggregated tables, *not* raw PII (no `email`, `phone`, `date_of_birth`)
- **`fraud_analysts`** — see masked PAN / Aadhaar, can see all rows including flagged ones, restricted to the fraud schema
- **`compliance`** — sees full PII for audit, but only customers in their assigned region

All three start with zero access; every GRANT in this notebook adds exactly what's needed.

## Managed vs. external tables — the deep dive

Notebook 02 introduced the distinction. The exam's Section 7 expects you to operate on it.

| Question | Managed | External |
|---|---|---|
| Who owns the files? | Unity Catalog | You |
| Storage location | UC-managed location (catalog or schema level) | A path you specify inside a UC **external location** |
| What does `DROP TABLE` do? | Drops metadata; files deleted after retention | Drops metadata only; files preserved |
| Predictive Optimization? | ✅ | ❌ |
| Lifecycle features (auto-OPTIMIZE, auto-VACUUM, deletion vectors fully managed)? | ✅ | Partial |
| When? | Default — the team produces and owns the data | Files must be shared with non-Databricks tools, or live at a regulator-mandated path |

**Creating a managed table** — no `LOCATION` clause:

```sql
 CREATE TABLE fintech_dev.silver.customers (
   customer_id STRING, ...
 ) USING DELTA;
```

**Creating an external table** — `LOCATION` points at a path inside a UC external location:

```sql
 CREATE TABLE fintech_dev.silver.customers (
   customer_id STRING, ...
 ) USING DELTA
 LOCATION 's3://bank-shared/silver/customers/';
```

**External locations are themselves UC objects** — registered once, with credentials managed via storage credentials. Reusing the same external location across many tables centralises permissions.

## Converting between managed and external

You can flip a table between managed and external **without rewriting data** via `ALTER TABLE ... SET MANAGED` / `SET EXTERNAL`. The exam expects you to know this is possible and what changes.

```sql
 -- Make a managed table external (UC keeps metadata, stops owning files)
 ALTER TABLE fintech_dev.silver.customers SET EXTERNAL;

 -- Adopt an external table as managed (UC takes ownership)
 ALTER TABLE fintech_dev.silver.customers SET MANAGED;
```

**Three exam-worthy facts:**

- The **three-part name** doesn't change; queries against the table continue to work.
- Whether files **physically move** depends on the direction and storage layout. Test on a copy first.
- The principal performing the conversion needs **MODIFY** on the table *and* permissions on the target storage / external location.

**Convert FROM external TO managed** is the more common direction — you started with files dropped by a third party and now want UC to manage their lifecycle (including Predictive Optimization).

## The security hierarchy

Unity Catalog's permission model is a strict tree. Privileges granted at any level **inherit** to everything below.

```text
  metastore
   └── catalog                    ← GRANT here inherits to ALL schemas / tables / views
        └── schema                ← GRANT here inherits to ALL tables / views in the schema
             └── table / view / volume / function / model / materialized view
```

**Two privileges that gate access at every level above an object:**

- **`USE CATALOG`** on a catalog — without it, you can't even *see* schemas inside.
- **`USE SCHEMA`** on a schema — without it, you can't see tables inside.

A grant of `SELECT` on `silver.customers` alone is useless if the principal lacks `USE CATALOG` on `fintech_dev` and `USE SCHEMA` on `silver`. **This is the most common exam trap** — the answer that omits `USE CATALOG` / `USE SCHEMA` is wrong.

**Inheritance shortcut.** `GRANT SELECT ON SCHEMA fintech_dev.silver` gives `SELECT` on every existing *and future* table in `silver`. Powerful, but also a foot-gun — prefer narrower grants for production.

## Privileges — what each one buys you

The list the exam expects you to recognise:

| Privilege | Applies to | Lets the grantee… |
|---|---|---|
| `USE CATALOG` | catalog | see schemas inside the catalog |
| `USE SCHEMA` | schema | see objects inside the schema |
| `SELECT` | table / view / MV | read rows |
| `MODIFY` | table / view | INSERT / UPDATE / DELETE / MERGE rows; ALTER for schema changes |
| `CREATE` | catalog / schema | create new objects inside |
| `CREATE TABLE`, `CREATE SCHEMA` etc. | parent | finer-grained create privileges |
| `EXECUTE` | function | call the function |
| `READ VOLUME` / `WRITE VOLUME` | volume | read or write files in the volume |
| `BROWSE` | catalog / schema | see metadata (names, comments) without seeing data |
| `APPLY TAG` | object | tag for governance (used by ABAC) |
| `ALL PRIVILEGES` | any | every privilege at that level (use sparingly) |

Ownership is separate from privileges. The **owner** of an object can do anything to it, including changing its grants. There's exactly **one owner** per object — `ALTER ... OWNER TO <group_or_principal>` transfers it.

## `GRANT`, `REVOKE`, `DENY` — on principals

Three SQL statements, three roles in the system.

**`GRANT`** adds a privilege:

```sql
 GRANT USE CATALOG ON CATALOG fintech_dev TO `analysts`;
 GRANT USE SCHEMA  ON SCHEMA  fintech_dev.gold TO `analysts`;
 GRANT SELECT      ON TABLE   fintech_dev.gold.customer_360 TO `analysts`;
```

**`REVOKE`** removes a privilege explicitly granted earlier:

```sql
 REVOKE SELECT ON TABLE fintech_dev.silver.customers FROM `analysts`;
```

**`DENY`** explicitly blocks a privilege, *even if some other grant would have allowed it*:

```sql
 DENY SELECT ON TABLE fintech_dev.silver.customers TO `analysts`;
```

**The precedence rule the exam tests:** **DENY beats GRANT.** If you granted `SELECT ON CATALOG fintech_dev TO analysts` but also denied `SELECT ON TABLE fintech_dev.silver.customers TO analysts`, the deny wins for that specific table.

**Three principal types:**

- **Users** — human accounts.
- **Groups** — preferred for production grants. Manage memberships in your identity provider; grant to the group name in UC.
- **Service principals** — non-human identities for jobs and CI/CD (notebook 07).

**Best practice the exam rewards:** grant to **groups**, never to individual users.

## Column masking

A **column mask** is a function that runs at query time on a specific column. The function decides — typically based on the current user / group — whether to return the real value or a masked version.

Two steps. First, declare the mask as a UC function.

```sql
 CREATE OR REPLACE FUNCTION fintech_dev.security.mask_pan(pan STRING)
 RETURNS STRING
 RETURN CASE
   WHEN is_account_group_member('fraud_analysts') THEN concat('XXXX-XXXX-XXXX-', right(pan, 4))
   WHEN is_account_group_member('compliance')     THEN pan       -- full access
   ELSE 'REDACTED'
 END;
```

Second, attach the mask to a column on the table.

```sql
 ALTER TABLE fintech_dev.silver.card_accounts
   ALTER COLUMN pan SET MASK fintech_dev.security.mask_pan;
```

After that, every read of `card_accounts.pan` goes through `mask_pan`. The function has full access to `current_user()`, `is_account_group_member(...)`, and other security functions.

**To remove a mask:** `ALTER TABLE ... ALTER COLUMN ... DROP MASK;`.

## Row filters

A **row filter** is a function that returns a boolean per row; UC adds a `WHERE` on every read against the table, filtering down to rows where the function returns `TRUE` for the querying principal.

```sql
 -- The function: which region is this user allowed to see?
 CREATE OR REPLACE FUNCTION fintech_dev.security.region_filter(country STRING)
 RETURNS BOOLEAN
 RETURN CASE
   WHEN is_account_group_member('compliance_in') THEN country = 'IN'
   WHEN is_account_group_member('compliance_us') THEN country = 'US'
   WHEN is_account_group_member('compliance')    THEN TRUE
   ELSE FALSE
 END;

 -- Attach to the table
 ALTER TABLE fintech_dev.silver.customers
   SET ROW FILTER fintech_dev.security.region_filter ON (country);
```

Now every `SELECT FROM fintech_dev.silver.customers` is implicitly `WHERE region_filter(country) = TRUE`. The querying principal sees only their slice.

**To remove a row filter:** `ALTER TABLE ... DROP ROW FILTER;`.

**Row filter + column mask = the bank's full PII regime.** A compliance analyst in India sees full PII for Indian customers only. An analyst sees only `country = 'IN'` rows *and* gets `REDACTED` in the PAN column.

## Unity Catalog ABAC — attribute-based access control

Per-table row filters and column masks work, but they don't scale to *hundreds* of tables. **Unity Catalog ABAC** (attribute-based access control) lets you define a policy *once* and apply it everywhere a table or column carries a specific **tag**.

**Three building blocks:**

- **Tags** — labels you attach to catalogs, schemas, tables, or columns (e.g. `pii_class = high`, `data_domain = cards`, `sensitivity = restricted`).
- **Policies** — central declarations like "mask any column tagged `pii_class = high` for everyone outside the `compliance` group" or "filter rows where `data_residency != current user's region`".
- **Evaluation** — at query time, UC looks at the queried object's tags, finds matching ABAC policies, and applies them on top of the principal's GRANTs.

```sql
 -- Tag a column on every PII-bearing table
 ALTER TABLE fintech_dev.silver.customers ALTER COLUMN email
   SET TAGS ('pii_class' = 'high');

 -- Define one policy that masks every 'pii_class = high' column for non-compliance users
 CREATE POLICY fintech_dev.security.mask_high_pii
   ON COLUMN MATCH TAGS ('pii_class' = 'high')
   USING fintech_dev.security.mask_email(value)
   WHEN NOT is_account_group_member('compliance');
```

Add a new PII column to a new table, tag it `pii_class = high`, and the existing policy covers it — no per-table mask grant needed.

**The exam expects you to recognise ABAC as the right answer when the question describes "centralised row-level filtering and column masking for sensitive data across many objects."**

## Dynamic views — the older alternative

Before column masks and row filters became first-class, the bank's pattern was a **dynamic view** — a regular view whose SELECT uses `CASE WHEN is_account_group_member(...) ...` to mask columns and filter rows in one go.

```sql
 CREATE OR REPLACE VIEW fintech_dev.gold.customers_secure AS
 SELECT customer_id,
        CASE WHEN is_account_group_member('compliance') THEN email ELSE 'REDACTED' END AS email,
        CASE WHEN is_account_group_member('compliance') THEN phone ELSE 'REDACTED' END AS phone,
        city, country, created_at
 FROM   fintech_dev.silver.customers
 WHERE  CASE WHEN is_account_group_member('compliance_in') THEN country = 'IN'
             WHEN is_account_group_member('compliance')    THEN TRUE
             ELSE FALSE END;
```

**When to still use dynamic views:**

- Very simple cases.
- Compatibility with non-UC clients that don't know about masks.

**Why masks + row filters (and ABAC) beat dynamic views:**

- Apply to the **base table**, so every reader, every query path, every BI tool gets them — there's no "forgot to use the secure view" failure mode.
- Compose with regular grants instead of having to re-grant the view.
- ABAC scales them across many tables via tags.

## Audit log

Every UC permission grant, every table access, every cross-account read is recorded in the **Unity Catalog audit log**. Two things the exam may probe:

- The audit log is the **forensic source of truth** — "did anyone query `silver.customers.email` outside compliance hours?".
- The **`system.access` schema** in the system catalog exposes audit data as queryable tables — `system.access.audit`, `system.access.table_lineage`, etc.

```sql
 SELECT event_time, user_identity.email, action_name,
        request_params.full_name_arg AS object
 FROM   system.access.audit
 WHERE  event_time > current_timestamp() - INTERVAL 1 DAY
   AND  request_params.full_name_arg LIKE 'fintech_prod.silver.customers%'
 ORDER BY event_time DESC;
```

**Lineage** — `system.access.table_lineage` records which jobs/notebooks read which tables and wrote which tables, including column-level lineage. This is what powers the *Lineage* tab in the UC UI.

## The decision sheet

| Scenario | Pick |
|---|---|
| Default for a new table | **Managed** |
| Files need to be readable by non-Databricks tools at a specific path | **External** |
| Give analysts read access to gold tables | `GRANT USE CATALOG`, `USE SCHEMA`, `SELECT` |
| Block a specific table for a group despite catalog-level grant | **`DENY`** |
| Hide PAN from analysts but show last 4 to fraud team | **Column mask** |
| Restrict compliance analysts to their region's rows | **Row filter** |
| Apply masks / filters to every PII-tagged column across hundreds of tables | **Unity Catalog ABAC** |
| Forensic audit: who read this table last Tuesday? | **`system.access.audit`** |
| Track column-level read/write lineage | **`system.access.table_lineage`** + UC Lineage tab |

## Hands-on — the bank's permission regime

The cells below grant the bank's three groups (`analysts`, `fraud_analysts`, `compliance`) exactly what they need against the Fintech catalog. Run in a Databricks notebook attached to a SQL warehouse or DBR 14.x+ cluster, as a user with `MANAGE` on `fintech_dev`.

The groups don't have to exist for the syntax to be valid — these cells emit the SQL even if the principals aren't provisioned.

In [ ]:
spark.sql("USE CATALOG fintech_dev")
spark.sql("CREATE SCHEMA IF NOT EXISTS security")
print("Working catalog: fintech_dev. Security functions live in fintech_dev.security.")

In [ ]:
# 1) Hierarchy walk: USE CATALOG → USE SCHEMA → SELECT, scoped tightly per group.
for stmt in [
    # analysts: read-only gold only
    "GRANT USE CATALOG ON CATALOG fintech_dev TO `analysts`",
    "GRANT USE SCHEMA  ON SCHEMA  fintech_dev.gold TO `analysts`",
    "GRANT SELECT      ON SCHEMA  fintech_dev.gold TO `analysts`",

    # fraud_analysts: read silver + gold; can also call security functions
    "GRANT USE CATALOG ON CATALOG fintech_dev TO `fraud_analysts`",
    "GRANT USE SCHEMA  ON SCHEMA  fintech_dev.silver TO `fraud_analysts`",
    "GRANT USE SCHEMA  ON SCHEMA  fintech_dev.gold   TO `fraud_analysts`",
    "GRANT SELECT      ON SCHEMA  fintech_dev.silver TO `fraud_analysts`",
    "GRANT SELECT      ON SCHEMA  fintech_dev.gold   TO `fraud_analysts`",

    # compliance: full read across silver + gold, plus write on a quarantine table later
    "GRANT USE CATALOG ON CATALOG fintech_dev TO `compliance`",
    "GRANT USE SCHEMA  ON SCHEMA  fintech_dev.silver TO `compliance`",
    "GRANT SELECT      ON SCHEMA  fintech_dev.silver TO `compliance`",
]:
    try:
        spark.sql(stmt)
        print("  OK :", stmt)
    except Exception as e:
        print("  --:", stmt, "->", str(e).splitlines()[0])

In [ ]:
# 2) DENY beats GRANT — block analysts from a specific raw bronze table even if a later
#    blanket SELECT on the schema is added.
try:
    spark.sql("DENY SELECT ON TABLE fintech_dev.bronze.card_transactions_raw TO `analysts`")
    print("DENY applied — analysts now blocked from bronze.card_transactions_raw regardless of inherited grants.")
except Exception as e:
    print("DENY recipe shown (run on a workspace where the principal exists):", str(e).splitlines()[0])

In [ ]:
# 3) Column mask: PAN is fully visible only to compliance; fraud_analysts see last-4; others REDACTED.
spark.sql("""
    CREATE OR REPLACE FUNCTION fintech_dev.security.mask_pan(pan STRING)
    RETURNS STRING
    RETURN CASE
      WHEN is_account_group_member('compliance')     THEN pan
      WHEN is_account_group_member('fraud_analysts') THEN concat('XXXX-XXXX-XXXX-', substring(pan, length(pan)-3, 4))
      ELSE 'REDACTED'
    END
""")
spark.sql("DESCRIBE FUNCTION EXTENDED fintech_dev.security.mask_pan").show(truncate=False)

In [ ]:
# 4) Apply the mask to a column. Requires the column to exist; demo creates a tiny card_accounts table.
spark.sql("""
    CREATE TABLE IF NOT EXISTS fintech_dev.silver.card_accounts (
      card_id STRING, customer_id STRING, pan STRING, card_type STRING, status STRING
    ) USING DELTA
""")
try:
    spark.sql("""
        ALTER TABLE fintech_dev.silver.card_accounts
          ALTER COLUMN pan SET MASK fintech_dev.security.mask_pan
    """)
    print("Mask applied. Every read of pan now goes through fintech_dev.security.mask_pan.")
except Exception as e:
    print("Mask recipe shown — run on a UC-enabled workspace with column masking GA:", str(e).splitlines()[0])

In [ ]:
# 5) Row filter on silver.customers: per-region compliance + global compliance + others see nothing.
spark.sql("""
    CREATE OR REPLACE FUNCTION fintech_dev.security.region_filter(country STRING)
    RETURNS BOOLEAN
    RETURN CASE
      WHEN is_account_group_member('compliance_in') THEN country = 'IN'
      WHEN is_account_group_member('compliance_us') THEN country = 'US'
      WHEN is_account_group_member('compliance')    THEN TRUE
      ELSE FALSE
    END
""")
try:
    spark.sql("""
        ALTER TABLE fintech_dev.silver.customers
          SET ROW FILTER fintech_dev.security.region_filter ON (country)
    """)
    print("Row filter applied. Every read of customers is implicitly WHERE region_filter(country) = TRUE.")
except Exception as e:
    print("Row filter recipe shown — run on a UC-enabled workspace with row filtering GA:", str(e).splitlines()[0])

In [ ]:
# 6) ABAC sketch — tag PII columns once, define one policy, scale to every tagged column anywhere.
ABAC_RECIPE = '''
-- Step A: tag every PII-bearing column across silver and gold
ALTER TABLE fintech_dev.silver.customers ALTER COLUMN email
  SET TAGS ('pii_class' = 'high');
ALTER TABLE fintech_dev.silver.customers ALTER COLUMN phone
  SET TAGS ('pii_class' = 'high');
ALTER TABLE fintech_dev.silver.customers ALTER COLUMN date_of_birth
  SET TAGS ('pii_class' = 'high');

-- Step B: one policy covers ALL high-PII columns, forever
CREATE POLICY fintech_dev.security.mask_high_pii
  ON COLUMN MATCH TAGS ('pii_class' = 'high')
  USING fintech_dev.security.mask_redact(value)
  WHEN NOT is_account_group_member('compliance');

-- Step C: new PII column? Just tag it. No new grant or mask needed.
ALTER TABLE fintech_dev.silver.new_table ALTER COLUMN passport_no
  SET TAGS ('pii_class' = 'high');
'''
print(ABAC_RECIPE)

In [ ]:
# 7) Audit log query — who read silver.customers in the last day?
try:
    spark.sql("""
        SELECT event_time,
               user_identity.email AS user_email,
               action_name,
               request_params['full_name_arg'] AS object
        FROM   system.access.audit
        WHERE  event_time > current_timestamp() - INTERVAL 1 DAY
          AND  request_params['full_name_arg'] LIKE 'fintech_prod.silver.customers%'
        ORDER BY event_time DESC
        LIMIT 20
    """).show(truncate=False)
except Exception as e:
    print("Audit query recipe shown — system.access requires UC + access to system catalog:", str(e).splitlines()[0])

## Section 7 self-check

**1.** Which privilege does an analyst need on `fintech_dev` *in addition to* `SELECT` on the gold tables, in order to actually query them?

- A. `MODIFY`
- B. `USE CATALOG` (and `USE SCHEMA` on the schema)
- C. `CREATE`
- D. `ALL PRIVILEGES`

**2.** A principal has `GRANT SELECT ON CATALOG fintech_dev` but also `DENY SELECT ON TABLE fintech_dev.silver.customers`. Can they SELECT from that table?

- A. Yes, the GRANT wins
- B. No, DENY beats GRANT
- C. Only if they're a service principal
- D. Only via a view

**3.** The bank wants to hide the PAN column from analysts but show last-4 to fraud_analysts. Which UC feature is correct?

- A. A view selecting only non-PAN columns
- B. A DENY on the column
- C. A **column mask** function that branches on `is_account_group_member`
- D. Row filter

**4.** Compliance analysts in India should see only customers with `country = 'IN'`. Which UC feature applies?

- A. Column mask
- B. **Row filter**
- C. Dynamic view only
- D. CHECK constraint

**5.** The data platform team wants one central policy that masks every column tagged `pii_class = high` across hundreds of tables for everyone outside `compliance`. Which feature is correct?

- A. Per-table column masks
- B. Per-table dynamic views
- C. **Unity Catalog ABAC**
- D. CHECK constraints

<details><summary>Show answers</summary>

1. **B** — `USE CATALOG` + `USE SCHEMA` are required at every level above the table.
2. **B** — DENY always beats GRANT.
3. **C** — a column mask is the dedicated UC feature for this.
4. **B** — row filters scope rows per principal.
5. **C** — ABAC + tags is the scale answer.

</details>

## Course recap — where to go from here

You've now covered every section of the **Databricks Certified Data Engineer Associate** exam (May 2026 version):

1. Notebook 01 — Platform & Compute (Section 1, 6%)
2. Notebook 02 — Delta Lake & Unity Catalog Foundations (Section 1)
3. Notebook 03 — Data Ingestion (Section 2, 21%)
4. Notebook 04 — Transformations (Section 3 part 1, 22%)
5. Notebook 05 — Modeling, MVs, Streaming Tables, Lakeflow Spark Declarative Pipelines (Section 3 part 2)
6. Notebook 06 — Lakeflow Jobs (Section 4, 16%)
7. Notebook 07 — CI/CD: Git Folders, Automation Bundles, CLI (Section 5, 10%)
8. Notebook 08 — Troubleshooting, Monitoring & Optimization (Section 6, 10%)
9. Notebook 09 — Governance & Security (Section 7, 15%)

**Final exam tips:**

- The exam is **scenario-driven**, not syntax-driven. For every choice, ask: *what problem is this question describing, and which feature is purpose-built for that problem?*
- When in doubt between two answers, the more **managed / declarative / Databricks-recommended** one is usually correct (managed tables, Lakeflow Spark Declarative Pipelines, Predictive Optimization, AQE, serverless SQL warehouses, ABAC).
- Memorise current product names (Lakeflow Jobs, Lakeflow Spark Declarative Pipelines, Databricks Git Folders, Automation Bundles). Legacy names (Workflows, DLT, Repos, DABs) appear as distractors.
- For trigger / ingestion / compute questions, the decision sheets in notebooks 03, 06, and 01 are the patterns the exam tests directly.

Good luck on the exam.